In [22]:
# Cell 1: Imports and Constants

from dataclasses import dataclass
import random

#CONSTANTS
MAX_WALK_MIN = 25 #maximum walking amount of 25 minutes
PRICE_WEIGHT = 2.0 #Every dollar for the parking lot price = 2 minutes

In [23]:
# Cell 2: Graph Definition

# Nodes for intersections 
NODES = ["S","A","B","C","E","D"]  # S = start, D = final destination

# Directed edges with travel time (minutes)
TRAVEL_TIMES = {
    ("S","A"): 4,
    ("S","B"): 6,
    ("A","C"): 5,
    ("B","C"): 2,
    ("B","E"): 4,
    ("C","D"): 6,
    ("C","E"): 3,
    ("E","D"): 2,
    ("A","E"): 8
}

#An adjacency list that acts as a map that shows which nodes are directly reachable from each node based off the graph.
ADJ = {n: [] for n in NODES}
for (u,v) in TRAVEL_TIMES:
    ADJ[u].append(v)


In [24]:
# Cell 3: Parking Lot Definition and RL environment logic

# Parking lot information
@dataclass(frozen=True)
class Lot:
    name: str
    node: str
    price: float           # Price in dollars
    walk_min: float        # Walking time in minutes
    capacity: int
    occupied: int

    # Function to check if parking lot has spots available
    @property
    def available(self):
        return self.occupied < self.capacity

# Parking lots list
LOTS = [
    Lot("P2", "B", 3, 20, 10, 10),   # full -> invalid
    Lot("P3", "C", 4, 10, 10, 2),    # valid
    Lot("P4", "E", 10, 5, 10, 9),    # valid
]

LOTS_BY_NODE = {lot.node: lot for lot in LOTS}

# Get possible actions from a given state (node)
def get_actions(state):
    return ADJ[state]

# Check if a state is terminal (i.e., valid parking lot)
def is_terminal(state):
    if state not in LOTS_BY_NODE:
        return False
    lot = LOTS_BY_NODE[state]
    return lot.available and lot.walk_min <= MAX_WALK_MIN

# Step function to transition from one state to another based on an action
def step(state, action):
    if action not in ADJ[state]:
        raise ValueError(f"Invalid action {action} from state {state}")
    
    next_state = action
    travel_time = TRAVEL_TIMES[(state, action)]

    # Default reward: negative travel time
    reward = -travel_time
    done = False

    # Check if next state is a parking lot
    if next_state in LOTS_BY_NODE:
        lot = LOTS_BY_NODE[next_state]

        # Valid parking
        if lot.available and lot.walk_min <= MAX_WALK_MIN:
            reward += -(lot.walk_min + PRICE_WEIGHT * lot.price)
            done = True

        # Invalid parking (full or too far walking distance)
        else:
            reward += -100
            done = True

    return next_state, reward, done

In [25]:
# Cell 4: Q-table initialization
# Q-table that stores the learned value for each (state, action) pair
Q = {}

# Initialize all Q-values to 0.0 for every valid state-action pair in the graph
for state in NODES:
    Q[state] = {}
    for action in get_actions(state):
        Q[state][action] = 0.0

# Print initial Q-table
print("Initial Q-table:")
for state in Q:
    print(state, Q[state])

Initial Q-table:
S {'A': 0.0, 'B': 0.0}
A {'C': 0.0, 'E': 0.0}
B {'C': 0.0, 'E': 0.0}
C {'D': 0.0, 'E': 0.0}
E {'D': 0.0}
D {}


In [26]:
# Cell 5: Epsilon-greedy action selection
# Probability of choosing a random action (exploration)
epsilon = 0.2


# Function to choose an action using epsilon-greedy strategy
def choose_action(state):

    actions = get_actions(state)

    # If no actions available, return None
    if len(actions) == 0:
        return None

    # Exploration: choose random action
    if random.random() < epsilon:
        return random.choice(actions)

    # Exploitation: choose action with highest Q-value
    best_action = actions[0]
    best_value = Q[state][best_action]

    for action in actions:
        if Q[state][action] > best_value:
            best_value = Q[state][action]
            best_action = action

    return best_action

In [27]:
# Cell 6: Q-learning Training + update rule

# Hyperparameters
alpha = 0.1
gamma = 0.9
episodes = 5000
max_steps = 10 

# Training loop
for ep in range(episodes):
    state = "S"
    steps = 0

    # Select actions and update Q-values through training
    while steps < max_steps:
        action = choose_action(state)
        # If no actions available, break
        if action is None:
            break

        next_state, reward, done = step(state, action)

        # Update Q-value using the Q-learning formula
        if next_state in Q and len(Q[next_state]) > 0:
            max_future = max(Q[next_state].values())
        else:
            max_future = 0

        # Q-learning update rule
        Q[state][action] = Q[state][action] + alpha * (
            reward + gamma * max_future - Q[state][action]
        )

        # Update current state to next state
        state = next_state
        steps += 1

        if done:
            break

# Display results
print("\nTrained Q-table:")
for state in Q:
    print(state, Q[state])


Trained Q-table:
S {'A': -24.699999999999974, 'B': -105.99999999999994}
A {'C': -22.999999999999986, 'E': -32.99999999999997}
B {'C': 0.0, 'E': 0.0}
C {'D': 0.0, 'E': 0.0}
E {'D': 0.0}
D {}


In [28]:
# Cell 7: Test the learned policy and display output

# Test the learned policy
state = "S"
path = [state]
total_reward = 0
max_steps = 10
steps = 0

# Follow the best learned policy 
while steps < max_steps:
    # Get possible actions from current state
    actions = get_actions(state)

    # If no actions available, break
    if len(actions) == 0:
        break

    # Choose best learned action (greedy, no exploration)
    best_action = max(Q[state], key=Q[state].get)
    
    next_state, reward, done = step(state, best_action)
    
    # Update path and total reward
    path.append(next_state)
    total_reward += reward

    # Update current state to next state
    state = next_state
    steps += 1

    if done:
        break

# Display results
print("Learned path:", " -> ".join(path))
print("Final state:", state)
print("Total reward:", total_reward)

if state in LOTS_BY_NODE:
    print("Chosen parking lot:", LOTS_BY_NODE[state].name)
    print("Parking lot available:", LOTS_BY_NODE[state].available)
    print("Walking time:", LOTS_BY_NODE[state].walk_min)
    print("Parking price:", LOTS_BY_NODE[state].price)
else:
    print("No parking lot was chosen.")

Learned path: S -> A -> C
Final state: C
Total reward: -27.0
Chosen parking lot: P3
Parking lot available: True
Walking time: 10
Parking price: 4


The learned policy shows that the agent selects the parking lot that minimizes total cost based on travel time, walking time, and parking price. The Q-learning algorithm successfully learned the optimal route from the starting node by updating Q-values over multiple episodes.